In [2]:
import os
import glob
import re
import PyPDF2
import numpy as np
import pandas as pd
from typing import Any
from google.genai import types
from google import genai as gclint
import google.generativeai as genai
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, stop_after_attempt, wait_random_exponential
from langchain_text_splitters import RecursiveCharacterTextSplitter
import time
import asyncio
import nest_asyncio
nest_asyncio.apply()

In [3]:
client = gclint.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL_ID = "gemini-2.0-flash" 
text_embedding_model = "text-embedding-004"
documents = glob.glob("Imam_Nawawi/*")

In [4]:
generation_config = {
    "temperature": 1,
    "top_p": 0.95,
    "top_k": 40,
    "max_output_tokens": 8192,
    "response_mime_type": "text/plain",
}

model = genai.GenerativeModel(
    model_name="gemini-2.0-flash",
    generation_config=generation_config,
)

In [5]:
async def async_generate_response(prompt):
    chat_session = model.start_chat(history=[])
    response = await asyncio.to_thread(chat_session.send_message, prompt)
    return response.text

In [6]:
@retry(wait=wait_random_exponential(multiplier=1, max=120), stop=stop_after_attempt(4))
def get_embeddings(
    embedding_client: Any, embedding_model: str, text: str, output_dim: int = 768
) -> list[float]:
    try:
        response = embedding_client.models.embed_content(
            model=embedding_model,
            contents=[text],
            config=types.EmbedContentConfig(output_dimensionality=output_dim),
        )
        return [response.embeddings[0].values]
    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e):
            return None
        print(f"Error generating embeddings: {str(e)}")
        raise

In [7]:
def build_index(
    document_paths: list[str],
    embedding_client: Any,
    embedding_model: str,
    chunk_size: int = 1500,
    chunk_overlap: int = 200,
    separators: list[str] = [
        "\n\nḤadīth No.", "\n\n", "\n", ". ", ", ", " ", ""],
    output_dim: int = 768,
    sleep_between_embeddings: float = 1.0 # Seconds to wait (adjust as needed)
) -> pd.DataFrame:
    """
    Builds an index from PDF documents using RecursiveCharacterTextSplitter,
    INCLUDING DELAYS to manage API quotas.
    """
    all_chunks_data = []
    print(f"Using chunk_size={chunk_size}, chunk_overlap={chunk_overlap}")
    print(f"Waiting {sleep_between_embeddings} seconds between embedding calls.")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators
    )

    total_chunks_processed = 0
    total_embeddings_failed = 0

    for doc_path in document_paths:
        print(f"Processing document: {doc_path}")
        try:
            with open(doc_path, "rb") as file:
                pdf_reader = PyPDF2.PdfReader(file)
                doc_name = os.path.basename(doc_path)

                for page_num in range(len(pdf_reader.pages)):
                    page = pdf_reader.pages[page_num]
                    try:
                         page_text = page.extract_text()
                         if not page_text:
                             continue
                    except Exception as page_err:
                        print(f"  Error extracting text from page {page_num + 1}: {page_err}")
                        continue

                    chunks = text_splitter.split_text(page_text)
                    # print(f"  Page {page_num + 1}: Found {len(chunks)} chunks.") # Less verbose

                    for chunk_num, chunk_text in enumerate(chunks):
                        total_chunks_processed += 1
                        if len(chunk_text.strip()) < 20:
                             continue
                        time.sleep(sleep_between_embeddings)

                        embeddings = get_embeddings(
                            embedding_client, embedding_model, chunk_text, output_dim=output_dim
                        )

                        if embeddings is None:
                            total_embeddings_failed += 1
                            print(
                                f"  Warning: Embedding failed for chunk {chunk_num} on page {page_num + 1} (likely quota)"
                            )
                            # Optionally add a longer sleep after a failure
                            # time.sleep(5)
                            continue # Skip this chunk

                        chunk_info = {
                            "document_name": doc_name,
                            "page_number": page_num + 1,
                            "chunk_number": chunk_num,
                            "chunk_text": chunk_text,
                            "embeddings": embeddings,
                        }
                        all_chunks_data.append(chunk_info)

                        # Optional: Print progress less frequently
                        if total_chunks_processed % 50 == 0:
                            print(f"  Processed {total_chunks_processed} chunks...")


        except Exception as e:
            print(f"Error processing document {doc_path}: {str(e)}")
            continue

    if not all_chunks_data:
        raise ValueError("No chunks were successfully created and embedded. Check API Keys, Quotas, and PDF files.")

    print(f"\nFinished processing.")
    print(f"Total chunks attempted: {total_chunks_processed}")
    print(f"Total chunks successfully embedded: {len(all_chunks_data)}")
    print(f"Total embedding failures (skipped): {total_embeddings_failed}")
    return pd.DataFrame(all_chunks_data)


In [9]:
if os.path.exists("vector_db_mini_vertex.csv"):
    print("vector_db_mini_vertex.csv already exists")
    vector_db_mini_vertex = pd.read_csv("vector_db_mini_vertex.csv")
else:
    print("Building vector_db_mini_vertex.csv...")
    vector_db_mini_vertex = build_index(
        documents,
        embedding_client=client,
        embedding_model=text_embedding_model,
        chunk_size=1500,
        chunk_overlap=200
    )

Building vector_db_mini_vertex.csv...
Using chunk_size=1500, chunk_overlap=200
Waiting 1.0 seconds between embedding calls.
Processing document: Imam_Nawawi/Server_rooms.pdf

Finished processing.
Total chunks attempted: 15
Total chunks successfully embedded: 15
Total embedding failures (skipped): 0


In [10]:
vector_db_mini_vertex.tail(5)

,document_name,page_number,chunk_number,chunk_text,embeddings
10,Server_rooms.pdf,6,0,"lines)\n \nto\n \nthe\n \ncentral\n \n""Server""...","[[0.10008587, 0.008953217, -0.0647227, -0.0159..."
11,Server_rooms.pdf,6,1,■\n \nOn\n \nthe\n \nServer\n \nLoad\n \nDispl...,"[[0.10569678, 0.00500749, -0.065157756, 0.0017..."
12,Server_rooms.pdf,7,0,Arrow\n \ntwice\n \nto\n \ndecrease\n \nthe\n ...,"[[0.105651915, 0.014186258, -0.033513512, 0.01..."
13,Server_rooms.pdf,7,1,4.\n \nIncrease\n \nTraffic\n \nVolume\n \n(Ma...,"[[0.07891798, -0.008526407, -0.050584637, 0.00..."
14,Server_rooms.pdf,8,0,5.\n \nFailur e\n \nOutcome\n:\n \n○\n \nThe\n...,"[[0.066506274, 0.013523649, -0.040942088, -0.0..."


In [11]:
def get_relevant_chunks(
    query: str,
    vector_db: pd.DataFrame,
    embedding_client: Any,
    embedding_model: str,
    top_k: int = 3,
) -> str:
    try:
        query_embedding = get_embeddings(embedding_client, embedding_model, query)

        if query_embedding is None:
            return "Could not process query due to quota issues"

        similarities = [
            cosine_similarity(query_embedding, chunk_emb)[0][0]
            for chunk_emb in vector_db["embeddings"]
        ]

        top_indices = np.argsort(similarities)[-top_k:]
        relevant_chunks = vector_db.iloc[top_indices]

        context = []
        for _, row in relevant_chunks.iterrows():
            context.append(
                {
                    "document_name": row["document_name"],
                    "page_number": row["page_number"],
                    "chunk_number": row["chunk_number"],
                    "chunk_text": row["chunk_text"],
                }
            )

        return "\n\n".join(
            [
                f"[Page {chunk['page_number']}, Chunk {chunk['chunk_number']}]: {chunk['chunk_text']}"
                for chunk in context
            ]
        )

    except Exception as e:
        print(f"Error getting relevant chunks: {str(e)}")
        return "Error retrieving relevant chunks"

In [12]:
@retry(wait=wait_random_exponential(multiplier=1, max=120), stop=stop_after_attempt(4))
async def generate_answer(
    query: str, context: str
) -> str:
    """
    Generates an answer to the query based on the provided context using an LLM.
    Includes specific instructions for explaining Hadith.
    """
    try:
        if context in [
            "Could not process query due to quota issues",
            "Error retrieving relevant chunks",
        ]:
            return "Could not generate answer because context retrieval failed or hit quota limits."

        prompt = f"""You are an assistant knowledgeable about Imam Nawawi's 40 Hadith.
            Your task is to answer the user's question based *only* on the provided context below, which is extracted from the book "The 40 Hadith of Al-Imam An-Nawawi".

            Instructions:
            1. Read the context carefully.
            2. If the question asks to explain a specific Hadith (e.g., "explain Hadith No. 1"), synthesize the information from the context that pertains *only* to that specific Hadith number and its explanation/commentary.
            3. Provide a clear and concise explanation based *strictly* on the given text.
            4. Do *not* add any information, commentary, opinions, or interpretations that are not present in the provided context.
            5. If the provided context does *not* contain the information needed to answer the question (e.g., the requested Hadith number is not found in the context), explicitly state that the information is not available in the provided context. Do not attempt to guess or retrieve information from external knowledge.
            6. Structure your answer clearly.

            Context:
            ---
            {context}
            ---

            Question: {query}

            Answer:"""
        response = await async_generate_response(prompt)
        return response

    except types.StopCandidateException as stop_ex:
        print(f"Generation stopped: {stop_ex}")
        return "The generation process was stopped, possibly due to safety settings or other limits."
    except Exception as e:
        if "RESOURCE_EXHAUSTED" in str(e) or (hasattr(e, 'message') and "RESOURCE_EXHAUSTED" in e.message):
            print(f"Quota Error during generation: {e}")
            return "Could not generate answer due to API quota limits."
        elif "response.text" in str(e): # Example check if response structure is bad
            print(f"API Response Error: {e}. The response object might be malformed.")
            return "Error processing the response from the language model."
        else:
            print(f"Error generating answer: {str(e)}")
            return f"An unexpected error occurred while generating the answer: {str(e)}"

In [13]:
def transform_query(query: str) -> str:
    """
    Transforms natural language numbers (first, second, etc.) in a query
    to standard "Hadith No. X" format for better retrieval matching.
    """
    query_lower = query.lower()
    replacements = {
    "first hadith": "Hadith No. 1",
    "hadith number one": "Hadith No. 1",
    "hadith number 1": "Hadith No. 1",
    "ḥadīth no. 1": "Hadith No. 1",

    "second hadith": "Hadith No. 2",
    "hadith number two": "Hadith No. 2",
    "hadith number 2": "Hadith No. 2",
    "ḥadīth no. 2": "Hadith No. 2",

    "third hadith": "Hadith No. 3",
    "hadith number three": "Hadith No. 3",
    "hadith number 3": "Hadith No. 3",
    "ḥadīth no. 3": "Hadith No. 3",

    "fourth hadith": "Hadith No. 4",
    "hadith number four": "Hadith No. 4",
    "hadith number 4": "Hadith No. 4",
    "ḥadīth no. 4": "Hadith No. 4",

    "fifth hadith": "Hadith No. 5",
    "hadith number five": "Hadith No. 5",
    "hadith number 5": "Hadith No. 5",
    "ḥadīth no. 5": "Hadith No. 5",

    "sixth hadith": "Hadith No. 6",
    "hadith number six": "Hadith No. 6",
    "hadith number 6": "Hadith No. 6",
    "ḥadīth no. 6": "Hadith No. 6",

    "seventh hadith": "Hadith No. 7",
    "hadith number seven": "Hadith No. 7",
    "hadith number 7": "Hadith No. 7",
    "ḥadīth no. 7": "Hadith No. 7",

    "eighth hadith": "Hadith No. 8",
    "hadith number eight": "Hadith No. 8",
    "hadith number 8": "Hadith No. 8",
    "ḥadīth no. 8": "Hadith No. 8",

    "ninth hadith": "Hadith No. 9",
    "hadith number nine": "Hadith No. 9",
    "hadith number 9": "Hadith No. 9",
    "ḥadīth no. 9": "Hadith No. 9",

    "tenth hadith": "Hadith No. 10",
    "hadith number ten": "Hadith No. 10",
    "hadith number 10": "Hadith No. 10",
    "ḥadīth no. 10": "Hadith No. 10",

    "eleventh hadith": "Hadith No. 11",
    "hadith number eleven": "Hadith No. 11",
    "hadith number 11": "Hadith No. 11",
    "ḥadīth no. 11": "Hadith No. 11",

    "twelfth hadith": "Hadith No. 12",
    "hadith number twelve": "Hadith No. 12",
    "hadith number 12": "Hadith No. 12",
    "ḥadīth no. 12": "Hadith No. 12",

    "thirteenth hadith": "Hadith No. 13",
    "hadith number thirteen": "Hadith No. 13",
    "hadith number 13": "Hadith No. 13",
    "ḥadīth no. 13": "Hadith No. 13",

    "fourteenth hadith": "Hadith No. 14",
    "hadith number fourteen": "Hadith No. 14",
    "hadith number 14": "Hadith No. 14",
    "ḥadīth no. 14": "Hadith No. 14",

    "fifteenth hadith": "Hadith No. 15",
    "hadith number fifteen": "Hadith No. 15",
    "hadith number 15": "Hadith No. 15",
    "ḥadīth no. 15": "Hadith No. 15",

    "sixteenth hadith": "Hadith No. 16",
    "hadith number sixteen": "Hadith No. 16",
    "hadith number 16": "Hadith No. 16",
    "ḥadīth no. 16": "Hadith No. 16",

    "seventeenth hadith": "Hadith No. 17",
    "hadith number seventeen": "Hadith No. 17",
    "hadith number 17": "Hadith No. 17",
    "ḥadīth no. 17": "Hadith No. 17",

    "eighteenth hadith": "Hadith No. 18",
    "hadith number eighteen": "Hadith No. 18",
    "hadith number 18": "Hadith No. 18",
    "ḥadīth no. 18": "Hadith No. 18",

    "nineteenth hadith": "Hadith No. 19",
    "hadith number nineteen": "Hadith No. 19",
    "hadith number 19": "Hadith No. 19",
    "ḥadīth no. 19": "Hadith No. 19",

    "twentieth hadith": "Hadith No. 20",
    "hadith number twenty": "Hadith No. 20",
    "hadith number 20": "Hadith No. 20",
    "ḥadīth no. 20": "Hadith No. 20",

    "twenty-first hadith": "Hadith No. 21",
    "hadith number twenty-one": "Hadith No. 21",
    "hadith number 21": "Hadith No. 21",
    "ḥadīth no. 21": "Hadith No. 21",

    "twenty-second hadith": "Hadith No. 22",
    "hadith number twenty-two": "Hadith No. 22",
    "hadith number 22": "Hadith No. 22",
    "ḥadīth no. 22": "Hadith No. 22",

    "twenty-third hadith": "Hadith No. 23",
    "hadith number twenty-three": "Hadith No. 23",
    "hadith number 23": "Hadith No. 23",
    "ḥadīth no. 23": "Hadith No. 23",

    "twenty-fourth hadith": "Hadith No. 24",
    "hadith number twenty-four": "Hadith No. 24",
    "hadith number 24": "Hadith No. 24",
    "ḥadīth no. 24": "Hadith No. 24",

    "twenty-fifth hadith": "Hadith No. 25",
    "hadith number twenty-five": "Hadith No. 25",
    "hadith number 25": "Hadith No. 25",
    "ḥadīth no. 25": "Hadith No. 25",

    "twenty-sixth hadith": "Hadith No. 26",
    "hadith number twenty-six": "Hadith No. 26",
    "hadith number 26": "Hadith No. 26",
    "ḥadīth no. 26": "Hadith No. 26",

    "twenty-seventh hadith": "Hadith No. 27",
    "hadith number twenty-seven": "Hadith No. 27",
    "hadith number 27": "Hadith No. 27",
    "ḥadīth no. 27": "Hadith No. 27",

    "twenty-eighth hadith": "Hadith No. 28",
    "hadith number twenty-eight": "Hadith No. 28",
    "hadith number 28": "Hadith No. 28",
    "ḥadīth no. 28": "Hadith No. 28",

    "twenty-ninth hadith": "Hadith No. 29",
    "hadith number twenty-nine": "Hadith No. 29",
    "hadith number 29": "Hadith No. 29",
    "ḥadīth no. 29": "Hadith No. 29",

    "thirtieth hadith": "Hadith No. 30",
    "hadith number thirty": "Hadith No. 30",
    "hadith number 30": "Hadith No. 30",
    "ḥadīth no. 30": "Hadith No. 30",

    "thirty-first hadith": "Hadith No. 31",
    "hadith number thirty-one": "Hadith No. 31",
    "hadith number 31": "Hadith No. 31",
    "ḥadīth no. 31": "Hadith No. 31",

    "thirty-second hadith": "Hadith No. 32",
    "hadith number thirty-two": "Hadith No. 32",
    "hadith number 32": "Hadith No. 32",
    "ḥadīth no. 32": "Hadith No. 32",

    "thirty-third hadith": "Hadith No. 33",
    "hadith number thirty-three": "Hadith No. 33",
    "hadith number 33": "Hadith No. 33",
    "ḥadīth no. 33": "Hadith No. 33",

    "thirty-fourth hadith": "Hadith No. 34",
    "hadith number thirty-four": "Hadith No. 34",
    "hadith number 34": "Hadith No. 34",
    "ḥadīth no. 34": "Hadith No. 34",

    "thirty-fifth hadith": "Hadith No. 35",
    "hadith number thirty-five": "Hadith No. 35",
    "hadith number 35": "Hadith No. 35",
    "ḥadīth no. 35": "Hadith No. 35",

    "thirty-sixth hadith": "Hadith No. 36",
    "hadith number thirty-six": "Hadith No. 36",
    "hadith number 36": "Hadith No. 36",
    "ḥadīth no. 36": "Hadith No. 36",

    "thirty-seventh hadith": "Hadith No. 37",
    "hadith number thirty-seven": "Hadith No. 37",
    "hadith number 37": "Hadith No. 37",
    "ḥadīth no. 37": "Hadith No. 37",

    "thirty-eighth hadith": "Hadith No. 38",
    "hadith number thirty-eight": "Hadith No. 38",
    "hadith number 38": "Hadith No. 38",
    "ḥadīth no. 38": "Hadith No. 38",

    "thirty-ninth hadith": "Hadith No. 39",
    "hadith number thirty-nine": "Hadith No. 39",
    "hadith number 39": "Hadith No. 39",
    "ḥadīth no. 39": "Hadith No. 39",

    "fortieth hadith": "Hadith No. 40",
    "hadith number forty": "Hadith No. 40",
    "hadith number 40": "Hadith No. 40",
    "ḥadīth no. 40": "Hadith No. 40",
}


    transformed_query = query # Start with original
    for key, value in replacements.items():
        if key in query_lower:
            transformed_query = re.sub(r'(?i)' + re.escape(key), value, transformed_query, count=1)
            break

    return transformed_query

In [14]:
async def rag(
    question: str,
    vector_db: pd.DataFrame,
    embedding_client: Any,
    embedding_model: str,
    top_k: int,
) -> tuple[str | None, str | None]: # Return answer and source page/chunk info

    """
    Performs Retrieval-Augmented Generation.
    Transforms the query, retrieves relevant chunks, and generates an answer.
    Includes improved error handling for context retrieval.
    """
    generated_answer = "Error: Processing failed."
    info_source = "N/A"
    relevant_context = None # Initialize context

    try:
        print(f"Original question: {question}")
        transformed_question = transform_query(question)
        print(f"Transformed question for retrieval: {transformed_question}")

        relevant_context = get_relevant_chunks(
            transformed_question, vector_db, embedding_client, embedding_model, top_k=top_k
        )
        print(f"Retrieved context snippet: '{str(relevant_context)[:200]}...'") # Log retrieved context

        if not relevant_context or relevant_context in [
             "Could not process query due to quota issues",
             "Error retrieving relevant chunks"
        ]:
             print(f"Context retrieval failed or returned error: {relevant_context}")
             generated_answer = "Could not generate answer because context retrieval failed (possibly due to quota or no relevant chunks found in the incomplete index)."
             info_source = "Context Retrieval Failed"
             return generated_answer, info_source # Exit early
        try:
            first_line = relevant_context.split('\n\n')[0]
            if ':' in first_line and '[' in first_line and ']' in first_line:
                 info_source = first_line.split(':', 1)[0].strip('[] ') # Gets "Page X, Chunk Y"
            else:
                 info_source = "Retrieved Context (Format Unknown)"
                 print(f"Warning: Could not parse standard source format from context header: '{first_line}'")
        except Exception as context_parse_err:
            print(f"Could not parse source info from context header: {context_parse_err}")
            info_source = "Source Parsing Error"

        generated_answer = await generate_answer(
            question, relevant_context
        )
        if "quota" in generated_answer.lower():
             info_source = "Generation Failed (Quota)"

        return generated_answer, info_source

    except Exception as e:
        print(f"Error processing question '{question}': {str(e)}")
        error_context_info = f"(Context snippet before error: '{str(relevant_context)[:100]}...')" if relevant_context else "(No context retrieved before error)"
        generated_answer = f"Error processing question: {str(e)} {error_context_info}"
        return generated_answer, info_source # Return defaults or error info


In [ ]:
from IPython.display import Audio, Markdown, display
response, info_source = await rag(
    question="explain Ḥadīth No. 1",
    vector_db=vector_db_mini_vertex,
    embedding_client=client,  # For embedding generation
    embedding_model=text_embedding_model,  # For embedding model
    top_k=3,
)
display(Markdown(response))
print("Source: ",info_source)

Original question: I am at Server Room 1, how can i succees here?
Transformed question for retrieval: I am at Server Room 1, how can i succees here?
Retrieved context snippet: '[Page 3, Chunk 1]: from
 
their
 
viewpoint.
 
Server
 
Room
 
1
 
Goal
 
and
 
Targets
 
●
 
Goal:
 
Successfully
 
overload
 
the
 
server
 
using
 
the
 
three
 
control
 
panels.
 
●
 
Time
 
Limi...'


To succeed in Server Room 1, you must overload the server using the three control panels (IP Distribution, Request Frequency, and Traffic Volume) within the 5-minute time limit and without exceeding the $25 budget. The winning condition requires setting the panels to the exact values of 9, 9, 9.

Here's the strategy:

*   **Step 1: Set IP Distribution:** Target the IP Distribution Panel and click the Right Arrow exactly 7 times (starting from 2) to reach 9 users. Monitor the budget closely.
*   **Step 2: Set Request Frequency:** Target the Request Frequency Panel. Adjust the slider rightwards to the position corresponding to a value of 9.
*   **Step 3: Set Traffic Volume:** Target the Traffic Volume Panel. Adjust the slider rightwards to the position corresponding to a value of 9.
*   **Step 4: Verify & Win:** Confirm the settings are 9, 9, 9. Check the budget reads 2 ($25 - $23). Ensure the timer hasn't expired. The Server Load Display (Pie Chart) should turn solid red, indicating a successful overload.


Source:  Page 3, Chunk 1


In [ ]:
from IPython.display import Audio, Markdown, display
response, info_source = await rag(
    question="what did Ibn Rajab, writing on the subject of prejudice?",
    vector_db=vector_db_mini_vertex,
    embedding_client=client,  # For embedding generation
    embedding_model=text_embedding_model,  # For embedding model
    top_k=3,
)
display(Markdown(response))
print("Source: ",info_source)

Original question: I am at Server Room 2, how can i succees here?
Transformed question for retrieval: I am at Server Room 2, how can i succees here?
Retrieved context snippet: '[Page 4, Chunk 1]: The
 
Server
 
Load
 
Display
 
(Pie
 
Chart)
 
should
 
turn
 
solid
 
red,
 
indicating
 
a
 
successful
 
overload.
 
 
Server
 
Room
 
2
 
Goal
 
and
 
Targets
:
 
●
 
Goal:
 
S...'


To succeed in Server Room 2, you need to overload the server using the three control panels within a 5-minute time limit and without exceeding the $25 budget.

Here's the strategy for success (the 5,10,10 path):

*   **Step 1: Set IP Distribution:** Target the IP Distribution Panel and click the Right Arrow exactly 3 times (starting from 2) to reach 5 users. Monitor the budget closely as it drops with each user increase.
*   **Step 2: Set Request Frequency:** Target the Request Frequency Panel and adjust the slider rightwards to the position corresponding to a value of 10.
*   **Step 3: Set Traffic Volume:** Target the Traffic Volume Panel and adjust the slider rightwards to the position corresponding to a value of 10.
*   **Step 4: Verify & Win:** Confirm the settings are 5, 10, 10. Check the budget reads 4 ($25 - $21). Ensure the timer hasn't expired. The Server Load Display (Pie Chart) should turn solid red, indicating a successful overload.


Source:  Page 4, Chunk 1


In [15]:
from IPython.display import Audio, Markdown, display
response, info_source = await rag(
    question="what are the functions of niyyah?",
    vector_db=vector_db_mini_vertex,
    embedding_client=client,  # For embedding generation
    embedding_model=text_embedding_model,  # For embedding model
    top_k=3,
)
display(Markdown(response))
print("Source: ",info_source)

Original question: what are the functions of niyyah?
Transformed question for retrieval: what are the functions of niyyah?
Retrieved context snippet: '[Page 55, Chunk 0]: 46 to it of " subḥān Allāh " earns additional reward, the capacity of what is between the 
heaven and earth.  Thus, scholars have inferred tha t the praise of Allah for His perfect...'


According to the provided text, the *niyyah* (intention), which is primarily in the heart, serves two functions:

1.  To define the worship intended – for example, what specific prayer will one be performing at a given time, or whether a *ghusl* (bath) is one required for the lifting of impurity or is a sunnah, such as that preceding *iḥrām* for *ḥajj* or *‘umrah*.
2.  To discern whether the particular act is one of worship – i.e., performed for Allah, or is one of habit, custom or worldly motivation.


Source:  Page 55, Chunk 0
